# Magnetic Domain CNN Reconstruction Pipeline
### Physics-informed U-Net: XMCD scattering patterns → spin vector field M(x,y)
**ECE 228 Final Project**

This notebook trains an attention U-Net to reconstruct 2D magnetic domain spin vector fields
M(x,y) = (Mx, My, Mz) from simulated XMCD coherent scattering amplitude patterns |A(q)|.

**Pipeline:**
1. Generate synthetic single-layer spin vector fields (Bloch/Neel walls, tanh profile)
2. Compute XMCD forward scattering: A(q) = FFT2[ k̂·M ] at multiple incident angles
3. Train CNN: |A(q)| → M(x,y)
4. Evaluate and visualize reconstruction quality

> **Runtime:** ~15-20 min on Colab T4 GPU for full training run


In [60]:
# Check GPU
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✓ GPU detected: {gpus[0].name}")
    print("Training will be fast (~15-20 min)")
else:
    print("\n⚠ No GPU detected — go to Runtime > Change runtime type > T4 GPU")


TensorFlow: 2.20.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✓ GPU detected: /physical_device:GPU:0
Training will be fast (~15-20 min)


In [61]:
# Install dependencies (scipy only — TF and numpy are pre-installed on Colab)
!pip install scipy -q


In [62]:
import os, warnings, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── CONFIG ──────────────────────────────────────────────────────────────────
GRID_N     = 64      # spatial grid (NxN) — increase to 128 for paper-quality
N_TRAIN    = 4000    # training samples
N_VAL      = 500     # validation samples
N_TEST     = 200
#N_ANGLES = len(THETA_LIST)  # now 6 instead of 4     # test samples
N_ANGLES   = 6       # incident beam directions per sample
BATCH_SIZE = 32      # larger batch on GPU
EPOCHS     = 60      # full training run
LR         = 3e-3

# Physics (Fe L3-edge XMCD, ALS COSMIC geometry)
HC_EV_NM   = 1239.84
PHOTON_EV  = 707.0
LAMBDA_NM  = HC_EV_NM / PHOTON_EV
XMCD_ASYM  = 0.3
SAMPLE_NM  = 75.0

PARAM_RANGES = {
    'domain_width'    : (1.5, 5.0),
    'wall_smoothness' : (0.005, 0.05),
    'wall_multiplier' : (1.5, 4.0),
    'chirality'       : [-1, +1],
    'theta_bulk_deg'  : (0, 180),
    'phi_bulk_deg'    : (0, 360),
    'domain_contrast' : ['alternating', 'uniform', 'orthogonal'],
    'assignment_mode' : ['split', 'tiles'],
    'stripe_axis'     : ['x', 'y'],
    'theta_inc_deg'   : (15, 60),
    'phi_inc_deg'     : (0, 360),
}

print("Config loaded.")
print(f"  Grid: {GRID_N}x{GRID_N}  |  Train: {N_TRAIN}  |  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}")


Config loaded.
  Grid: 64x64  |  Train: 4000  |  Epochs: 60  |  Batch: 32


## Section 1 — Spin Vector Field Generator
Python port of `module1.m`. Generates M(x,y) = (Mx, My, Mz) with Bloch/Neel walls, tanh profile, tunable chirality and domain contrast.

In [63]:
def sample_params(rng):
    p = {}
    p['domain_width']    = rng.uniform(*PARAM_RANGES['domain_width'])
    p['wall_smoothness'] = rng.uniform(*PARAM_RANGES['wall_smoothness'])
    p['wall_multiplier'] = rng.uniform(*PARAM_RANGES['wall_multiplier'])
    p['chirality']       = rng.choice(PARAM_RANGES['chirality'])
    p['theta_bulk_deg']  = rng.uniform(*PARAM_RANGES['theta_bulk_deg'])
    p['phi_bulk_deg']    = rng.uniform(*PARAM_RANGES['phi_bulk_deg'])
    p['domain_contrast'] = rng.choice(PARAM_RANGES['domain_contrast'])
    p['assignment_mode'] = rng.choice(PARAM_RANGES['assignment_mode'])
    p['stripe_axis']     = rng.choice(PARAM_RANGES['stripe_axis'])
    return p

def generate_domain_field(N, params, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    x_range = (-10, 10); y_range = (-10, 10)
    x = np.linspace(x_range[0], x_range[1], N)
    y = np.linspace(y_range[0], y_range[1], N)
    X, Y = np.meshgrid(x, y)

    dw   = params['domain_width']
    ws   = params['wall_smoothness']
    wm   = params['wall_multiplier']
    chir = params['chirality']
    theta_bulk = np.deg2rad(params['theta_bulk_deg'])
    phi_bulk   = np.deg2rad(params['phi_bulk_deg'])
    contrast   = params['domain_contrast']
    assign     = params['assignment_mode']
    axis       = params['stripe_axis']
    ws_eff = ws * wm

    S    = X if axis == 'x' else Y
    Smin = x_range[0] if axis == 'x' else y_range[0]
    domain_index = np.floor((S - Smin) / max(dw, 1e-9)).astype(int)
    s_mod = (S - Smin) % max(dw, 1e-9) - max(dw, 1e-9) / 2
    fk = np.tanh(s_mod / max(ws_eff, 1e-6))
    wall_mask = np.abs(s_mod) < 0.35

    theta_dom1 = theta_bulk
    if contrast == 'uniform':
        theta_dom2 = theta_bulk
    elif contrast == 'orthogonal':
        theta_dom2 = theta_bulk + np.pi / 2
    else:
        theta_dom2 = theta_bulk + np.pi

    is_odd = (domain_index % 2) == 1
    theta_base = np.where(is_odd, theta_dom2, theta_dom1)

    if assign == 'tiles':
        tile_N = 6
        tx = np.clip(((X - x_range[0]) / (x_range[1]-x_range[0]) * tile_N).astype(int), 0, tile_N-1)
        ty = np.clip(((Y - y_range[0]) / (y_range[1]-y_range[0]) * tile_N).astype(int), 0, tile_N-1)
        bloch_mask = ((tx + ty) % 2) == 0
    else:
        bloch_mask = X >= rng.uniform(-2, 2)
    neel_mask = ~bloch_mask

    epsilon = np.pi / 4
    Nlayers = max(1, round(np.pi / abs(epsilon)))
    Mx = np.zeros((N, N)); My = np.zeros((N, N)); Mz = np.zeros((N, N))

    for k in range(1, Nlayers + 1):
        theta_kz  = theta_base + chir * k * epsilon * fk
        in_plane  = np.sin(theta_kz)
        out_plane = np.cos(theta_kz)
        Mx[neel_mask]  += in_plane[neel_mask]  * np.cos(phi_bulk)
        My[neel_mask]  += in_plane[neel_mask]  * np.sin(phi_bulk)
        Mz[neel_mask]  += out_plane[neel_mask]
        phi_bloch = phi_bulk + np.pi / 2
        Mx[bloch_mask] += in_plane[bloch_mask] * np.cos(phi_bloch)
        My[bloch_mask] += in_plane[bloch_mask] * np.sin(phi_bloch)
        Mz[bloch_mask] += out_plane[bloch_mask]

    mag = np.sqrt(Mx**2 + My**2 + Mz**2); mag[mag < 1e-12] = 1.0
    return Mx/mag, My/mag, Mz/mag, wall_mask, neel_mask, bloch_mask

print("Vector field generator ready.")


Vector field generator ready.


## Section 2 — XMCD Forward Scattering Model
Single-layer Born approximation: A(q) = FFT2[ k̂·M ] × XMCD asymmetry

In [64]:
def forward_scatter(Mx, My, Mz, theta_inc_deg, phi_inc_deg, xmcd=XMCD_ASYM):
    theta_r = np.deg2rad(theta_inc_deg)
    phi_r   = np.deg2rad(phi_inc_deg)
    k_hat = np.array([
        np.sin(theta_r) * np.cos(phi_r),
        np.sin(theta_r) * np.sin(phi_r),
        -np.cos(theta_r)
    ])
    proj = k_hat[0]*Mx + k_hat[1]*My + k_hat[2]*Mz
    F    = np.fft.fftshift(np.fft.fft2(proj))
    return np.abs(F) * xmcd, np.angle(F), proj

# Azimuthal integration parameters
N_PHI_INT   = 36    # phi samples per theta (every 10 degrees)
THETA_LIST  = [15, 25, 35, 45, 55, 60]  # 6 polar angles — one channel each

def build_multi_angle_input(Mx, My, Mz, log_scale=True):
    """
    For each polar angle theta, sum amplitude over all phi angles.
    Output: [N, N, N_THETA] — one channel per polar angle.

    This gives ~216 projections worth of information in 6 channels,
    making the inverse problem far better constrained.
    """
    phi_list = np.linspace(0, 360, N_PHI_INT, endpoint=False)
    channels = []

    for theta in THETA_LIST:
        amp_sum = None
        for phi in phi_list:
            amp, _, _ = forward_scatter(Mx, My, Mz, theta, phi)
            if amp_sum is None:
                amp_sum = amp
            else:
                amp_sum = amp_sum + amp
        # normalize
        if log_scale:
            amp_sum = np.log1p(amp_sum)
        amp_max = amp_sum.max()
        if amp_max > 1e-12:
            amp_sum = amp_sum / amp_max
        channels.append(amp_sum)

    return np.stack(channels, axis=-1)   # [N, N, 6]

print("Forward scattering model ready.")
print(f"  {len(THETA_LIST)} polar angles x {N_PHI_INT} phi samples = {len(THETA_LIST)*N_PHI_INT} projections per sample")
print(f"  Output channels per sample: {len(THETA_LIST)}")

Forward scattering model ready.
  6 polar angles x 36 phi samples = 216 projections per sample
  Output channels per sample: 6


## Section 3 — Dataset Generation

In [65]:
def generate_dataset(n_samples, seed=42, N=GRID_N):
    rng  = np.random.default_rng(seed)
    X    = np.zeros((n_samples, N, N, len(THETA_LIST)), dtype=np.float32)
    Y    = np.zeros((n_samples, N, N, 3),               dtype=np.float32)
    meta = []
    t0   = time.time()
    for i in range(n_samples):
        params = sample_params(rng)
        Mx, My, Mz, wall_mask, neel_mask, bloch_mask = generate_domain_field(N, params, rng)
        X[i] = build_multi_angle_input(Mx, My, Mz)
        Y[i, :, :, 0] = Mx.astype(np.float32)
        Y[i, :, :, 1] = My.astype(np.float32)
        Y[i, :, :, 2] = Mz.astype(np.float32)
        meta.append({'params': params,
                     'wall_frac': wall_mask.mean(),
                     'bloch_frac': bloch_mask.mean()})
        if (i+1) % 500 == 0:
            rate = (i+1)/(time.time()-t0)
            print(f"  {i+1}/{n_samples}  ({rate:.0f} samples/s  ETA {(n_samples-i-1)/rate:.0f}s)")
    return X, Y, meta

print("Generating training data...")
t0 = time.time()
X_train, Y_train, meta_train = generate_dataset(N_TRAIN, seed=42)
X_val,   Y_val,   meta_val   = generate_dataset(N_VAL,   seed=99)
X_test,  Y_test,  meta_test  = generate_dataset(N_TEST,  seed=137)
print(f"Done in {time.time()-t0:.1f}s")
print(f"X_train: {X_train.shape}  Y_train: {Y_train.shape}")


Generating training data...
  500/4000  (18 samples/s  ETA 190s)
  1000/4000  (19 samples/s  ETA 162s)
  1500/4000  (19 samples/s  ETA 135s)
  2000/4000  (18 samples/s  ETA 108s)
  2500/4000  (19 samples/s  ETA 81s)
  3000/4000  (19 samples/s  ETA 54s)
  3500/4000  (19 samples/s  ETA 27s)
  4000/4000  (19 samples/s  ETA 0s)
  500/500  (19 samples/s  ETA 0s)
Done in 250.7s
X_train: (4000, 64, 64, 6)  Y_train: (4000, 64, 64, 3)


## Section 4 — Attention U-Net Architecture
16M parameter encoder-decoder with residual blocks and attention gates.

In [66]:
def residual_block(x, filters, name_prefix=''):
    skip = x
    if x.shape[-1] != filters:
        skip = layers.Conv2D(filters, 1, padding='same', name=f'{name_prefix}_skip')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{name_prefix}_c1')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = layers.Activation('gelu', name=f'{name_prefix}_act1')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{name_prefix}_c2')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
    x = layers.Add(name=f'{name_prefix}_add')([x, skip])
    return layers.Activation('gelu', name=f'{name_prefix}_act2')(x)

def attention_gate(x, g, filters, name_prefix=''):
    theta = layers.Conv2D(filters, 1, padding='same', name=f'{name_prefix}_theta')(x)
    phi   = layers.Conv2D(filters, 1, padding='same', name=f'{name_prefix}_phi')(g)
    add   = layers.Add(name=f'{name_prefix}_add')([theta, phi])
    relu  = layers.Activation('relu', name=f'{name_prefix}_relu')(add)
    psi   = layers.Conv2D(1, 1, activation='sigmoid', name=f'{name_prefix}_psi')(relu)
    return layers.Multiply(name=f'{name_prefix}_mult')([x, psi])

def build_model(N=GRID_N, n_angles=N_ANGLES):
    inputs = keras.Input(shape=(N, N, n_angles), name='scattering_input')
    e1 = residual_block(inputs, 32,  'e1a'); e1 = residual_block(e1, 32,  'e1b')
    p1 = layers.MaxPooling2D(name='pool1')(e1)
    e2 = residual_block(p1,    64,  'e2a'); e2 = residual_block(e2, 64,  'e2b')
    p2 = layers.MaxPooling2D(name='pool2')(e2)
    e3 = residual_block(p2,    128, 'e3a'); e3 = residual_block(e3, 128, 'e3b')
    p3 = layers.MaxPooling2D(name='pool3')(e3)
    e4 = residual_block(p3,    256, 'e4a'); e4 = residual_block(e4, 256, 'e4b')
    p4 = layers.MaxPooling2D(name='pool4')(e4)
    b  = residual_block(p4,    512, 'b1');  b  = residual_block(b,  512, 'b2')

    u4 = layers.UpSampling2D(name='up4')(b)
    u4 = layers.Concatenate(name='cat4')([u4, attention_gate(e4, u4, 128, 'ag4')])
    d4 = residual_block(u4, 256, 'd4a'); d4 = residual_block(d4, 256, 'd4b')

    u3 = layers.UpSampling2D(name='up3')(d4)
    u3 = layers.Concatenate(name='cat3')([u3, attention_gate(e3, u3, 64, 'ag3')])
    d3 = residual_block(u3, 128, 'd3a'); d3 = residual_block(d3, 128, 'd3b')

    u2 = layers.UpSampling2D(name='up2')(d3)
    u2 = layers.Concatenate(name='cat2')([u2, attention_gate(e2, u2, 32, 'ag2')])
    d2 = residual_block(u2, 64,  'd2a'); d2 = residual_block(d2, 64,  'd2b')

    u1 = layers.UpSampling2D(name='up1')(d2)
    u1 = layers.Concatenate(name='cat1')([u1, attention_gate(e1, u1, 16, 'ag1')])
    d1 = residual_block(u1, 32,  'd1a'); d1 = residual_block(d1, 32,  'd1b')

    out = layers.Conv2D(3, 1, activation='tanh', name='output')(d1)
    return keras.Model(inputs, out, name='MagCNN')

model = build_model(N=GRID_N, n_angles=len(THETA_LIST))
print(f"Model built. Parameters: {model.count_params():,}")


Model built. Parameters: 16,288,375


## Section 5 — Physics-Informed Loss Functions

In [67]:
def combined_loss(y_true, y_pred):
    return tf.reduce_mean((y_true - y_pred) ** 2)

@tf.function
def train_step(Xb, Yb):
    with tf.GradientTape() as tape:
        Yp = model(Xb, training=True)
        loss = combined_loss(Yb, Yp)
    g = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(g, model.trainable_variables))
    return loss

@tf.function
def val_step(Xb, Yb):
    Yp = model(Xb, training=False)
    return combined_loss(Yb, Yp)


## Section 6 — Training
This cell runs the full training loop. On a Colab T4 GPU expect ~15-20 minutes.

In [ ]:
optimizer = keras.optimizers.Adam(LR)
best_val  = np.inf
best_weights = None

@tf.function
def train_step(Xb, Yb):
    with tf.GradientTape() as tape:
        Yp = model(Xb, training=True)
        loss = tf.reduce_mean((Yb - Yp) ** 2)
    g = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(g, model.trainable_variables))
    return loss

@tf.function
def val_step(Xb, Yb):
    Yp = model(Xb, training=False)
    return tf.reduce_mean((Yb - Yp) ** 2)

print(f"Training for {EPOCHS} epochs  |  {N_TRAIN} samples  |  batch {BATCH_SIZE}")
print("="*60)
t_start = time.time()
history = {'train_loss': [], 'val_loss': []}

for ep in range(1, EPOCHS + 1):
    idx = np.random.permutation(N_TRAIN)
    tl = []
    for b in range(0, N_TRAIN, BATCH_SIZE):
        ib = idx[b:b+BATCH_SIZE]
        loss = train_step(tf.constant(X_train[ib], tf.float32),
                          tf.constant(Y_train[ib], tf.float32))
        tl.append(float(loss))

    vl = []
    for b in range(0, N_VAL, BATCH_SIZE):
        loss = val_step(tf.constant(X_val[b:b+BATCH_SIZE], tf.float32),
                        tf.constant(Y_val[b:b+BATCH_SIZE], tf.float32))
        vl.append(float(loss))

    history['train_loss'].append(np.mean(tl))
    history['val_loss'].append(np.mean(vl))

    if np.mean(vl) < best_val:
        best_val = np.mean(vl)
        best_weights = [w.numpy() for w in model.weights]

    elapsed = time.time() - t_start
    eta = (elapsed / ep) * (EPOCHS - ep)
    print(f"Ep {ep:3d}/{EPOCHS} | loss {np.mean(tl):.4f} | val_loss {np.mean(vl):.4f} | {elapsed/60:.1f}min (ETA {eta/60:.1f}min)")

if best_weights:
    for w, v in zip(model.weights, best_weights):
        w.assign(v)
print(f"\nDone. Best val loss: {best_val:.4f}")


## Section 7 — Evaluation

In [ ]:
eps = 1e-7
preds, angs, l2s = [], [], []
for b in range(0, N_TEST, BATCH_SIZE):
    Yp = model(tf.constant(X_test[b:b+BATCH_SIZE], tf.float32), training=False).numpy()
    Yb = Y_test[b:b+BATCH_SIZE]
    preds.append(Yp)
    nt  = np.linalg.norm(Yb, axis=-1, keepdims=True) + eps
    np_ = np.linalg.norm(Yp, axis=-1, keepdims=True) + eps
    cos = np.clip(np.sum((Yb/nt)*(Yp/np_), axis=-1), -1+eps, 1-eps)
    angs.append(np.degrees(np.arccos(cos)).mean(axis=(1,2)))
    l2s.append([100*np.linalg.norm((Yb[i]-Yp[i]).ravel())/(np.linalg.norm(Yb[i].ravel())+eps)
                for i in range(len(Yb))])

preds = np.concatenate(preds)
angs  = np.concatenate(angs)
l2s   = np.concatenate(l2s)

print("="*54)
print("  TEST SET RESULTS")
print("="*54)
print(f"  Mean angular error   : {angs.mean():.2f}°  (random baseline: 90°)")
print(f"  Median angular error : {np.median(angs):.2f}°")
print(f"  Std angular error    : {angs.std():.2f}°")
print(f"  Mean L2 error        : {l2s.mean():.2f}%")
print(f"  Median L2 error      : {np.median(l2s):.2f}%")
print("="*54)


## Section 8 — Figures

In [ ]:
# Fig 1 — Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9')
    for s in ax.spines.values(): s.set_color('#30363d')

ep_range = range(1, len(history['train_ang'])+1)
axes[0].plot(ep_range, history['train_ang'], color='#58a6ff', lw=2, label='Train')
axes[0].plot(ep_range, history['val_ang'],   color='#f78166', lw=2, label='Val')
axes[0].axhline(90, color='#8b949e', ls='--', lw=1, label='Random (90°)')
axes[0].set_xlabel('Epoch', color='#c9d1d9'); axes[0].set_ylabel('Angular Error (°)', color='#c9d1d9')
axes[0].set_title('Mean Angular Error', color='#e6edf3', fontweight='bold')
axes[0].legend(framealpha=0.3, labelcolor='#c9d1d9', facecolor='#161b22')

axes[1].plot(ep_range, history['train_loss'], color='#58a6ff', lw=2, label='Train')
axes[1].plot(ep_range, history['val_loss'],   color='#f78166', lw=2, label='Val')
axes[1].set_xlabel('Epoch', color='#c9d1d9'); axes[1].set_ylabel('Loss', color='#c9d1d9')
axes[1].set_title('Combined Loss', color='#e6edf3', fontweight='bold')
axes[1].legend(framealpha=0.3, labelcolor='#c9d1d9', facecolor='#161b22')

fig.suptitle('MagCNN Training Curves', color='#e6edf3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_training_curves.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved fig1_training_curves.png")


In [ ]:
# Fig 2 — Domain reconstruction examples (4 test samples)
n_ex = 4
fig  = plt.figure(figsize=(16, 4.5*n_ex))
fig.patch.set_facecolor('#0d1117')
gs   = gridspec.GridSpec(n_ex, 4, hspace=0.4, wspace=0.25)

for row in range(n_ex):
    Xs = X_test[row]; Yt = Y_test[row]; Yp = preds[row]
    N  = Xs.shape[0]
    step = max(1, N//9)
    xs = np.arange(0,N,step); ys = np.arange(0,N,step)
    XX, YY = np.meshgrid(xs, ys)
    eps = 1e-7
    nt  = np.linalg.norm(Yt,axis=-1,keepdims=True)+eps
    np_ = np.linalg.norm(Yp,axis=-1,keepdims=True)+eps
    cos = np.clip(np.sum((Yt/nt)*(Yp/np_),axis=-1),-1+eps,1-eps)
    ang_map = np.degrees(np.arccos(cos))

    for ci, (data, title, cmap, vmin, vmax) in enumerate([
        (Xs[:,:,0],   'Input |A(q)|',          plt.cm.inferno, None, None),
        (Yt[:,:,2],   'True Mz + (Mx,My)',      plt.cm.RdBu_r,  -1,   1),
        (Yp[:,:,2],   'Predicted Mz + (Mx,My)', plt.cm.RdBu_r,  -1,   1),
        (ang_map,     f'Ang Error (mean={ang_map.mean():.1f}°)', plt.cm.hot_r, 0, 90),
    ]):
        ax = fig.add_subplot(gs[row, ci])
        ax.set_facecolor('#161b22')
        for s in ax.spines.values(): s.set_color('#30363d')
        kw = dict(cmap=cmap, origin='lower')
        if vmin is not None: kw['vmin'] = vmin; kw['vmax'] = vmax
        im = ax.imshow(data, **kw)
        plt.colorbar(im, ax=ax, fraction=0.046)
        if ci in [1,2]:
            ax.quiver(XX, YY,
                      (Yt if ci==1 else Yp)[::step,::step,0],
                      (Yt if ci==1 else Yp)[::step,::step,1],
                      color='k', alpha=0.6, scale=14, width=0.005)
        ax.set_title(title, color='#e6edf3', fontsize=9)
        ax.axis('off')

fig.suptitle('MagCNN: Scattering Pattern → Spin Vector Field M(x,y)',
             color='#e6edf3', fontsize=13, fontweight='bold')
plt.savefig('fig2_reconstruction_examples.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Saved fig2_reconstruction_examples.png")


In [ ]:
# Fig 3 — Single sample domain reconstruction (paper figure)
idx  = 0
Xs   = X_test[idx]; Yt = Y_test[idx]; Yp = preds[idx]
meta = meta_test[idx]
N    = Xs.shape[0]
eps  = 1e-7
nt   = np.linalg.norm(Yt,axis=-1,keepdims=True)+eps
np_  = np.linalg.norm(Yp,axis=-1,keepdims=True)+eps
cos  = np.clip(np.sum((Yt/nt)*(Yp/np_),axis=-1),-1+eps,1-eps)
ang_map = np.degrees(np.arccos(cos))

step = max(1, N//10)
xs = np.arange(0,N,step); ys = np.arange(0,N,step)
XX, YY = np.meshgrid(xs, ys)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0d1117')
p = meta['params']

for i, (ax, data, title, cmap, vmin, vmax, qx, qy) in enumerate(zip(axes,
    [Yt[:,:,2],   Yp[:,:,2],   ang_map],
    ['True Domain Structure', 'CNN Reconstruction', 'Angular Error (°)'],
    [plt.cm.RdBu_r, plt.cm.RdBu_r, plt.cm.hot_r],
    [-1, -1, 0], [1, 1, 90],
    [Yt[::step,::step,0], Yp[::step,::step,0], None],
    [Yt[::step,::step,1], Yp[::step,::step,1], None],
)):
    ax.set_facecolor('#161b22')
    for s in ax.spines.values(): s.set_color('#30363d')
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, origin='lower')
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='#c9d1d9', fontsize=8)
    if qx is not None:
        ax.quiver(XX, YY, qx, qy, color='k', alpha=0.65, scale=12, width=0.005)
    ax.set_title(title, color='#e6edf3', fontsize=12, fontweight='bold')
    ax.set_xlabel('x (pixels)', color='#c9d1d9'); ax.set_ylabel('y (pixels)', color='#c9d1d9')
    ax.tick_params(colors='#c9d1d9')

fig.suptitle(
    f"Magnetic Domain Reconstruction — Sample #{idx}\n"
    f"dw={p['domain_width']:.2f}  contrast={p['domain_contrast']}  "
    f"walls={p['assignment_mode']}  chirality={p['chirality']:+d}\n"
    f"Mean angular error: {ang_map.mean():.1f}°   Median: {np.median(ang_map):.1f}°",
    color='#e6edf3', fontsize=11, fontweight='bold', y=1.04)

plt.tight_layout()
plt.savefig('fig3_domain_reconstruction.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

# Print metrics
print(f"\nSample #{idx} metrics:")
print(f"  domain_width   : {p['domain_width']:.3f}")
print(f"  contrast       : {p['domain_contrast']}")
print(f"  chirality      : {p['chirality']:+d}")
print(f"  Mean ang error : {ang_map.mean():.2f}°")
print(f"  Median ang err : {np.median(ang_map):.2f}°")
print(f"  L2 error       : {l2s[idx]:.2f}%")


In [ ]:
# Fig 4 — Error distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#c9d1d9')
    for s in ax.spines.values(): s.set_color('#30363d')

axes[0].hist(angs, bins=25, color='#58a6ff', alpha=0.85, edgecolor='#30363d')
axes[0].axvline(angs.mean(),     color='#f78166', lw=2, label=f'Mean {angs.mean():.1f}°')
axes[0].axvline(np.median(angs), color='#3fb950', lw=2, ls='--', label=f'Median {np.median(angs):.1f}°')
axes[0].axvline(90, color='#8b949e', lw=1, ls=':', label='Random baseline')
axes[0].set_xlabel('Mean Angular Error (°)', color='#c9d1d9')
axes[0].set_ylabel('Count', color='#c9d1d9')
axes[0].set_title('Angular Error Distribution', color='#e6edf3', fontweight='bold')
axes[0].legend(framealpha=0.3, labelcolor='#c9d1d9', facecolor='#161b22')

axes[1].hist(l2s, bins=25, color='#3fb950', alpha=0.85, edgecolor='#30363d')
axes[1].axvline(l2s.mean(),     color='#f78166', lw=2, label=f'Mean {l2s.mean():.1f}%')
axes[1].axvline(np.median(l2s), color='#58a6ff', lw=2, ls='--', label=f'Median {np.median(l2s):.1f}%')
axes[1].set_xlabel('Relative L2 Error (%)', color='#c9d1d9')
axes[1].set_ylabel('Count', color='#c9d1d9')
axes[1].set_title('L2 Error Distribution', color='#e6edf3', fontweight='bold')
axes[1].legend(framealpha=0.3, labelcolor='#c9d1d9', facecolor='#161b22')

fig.suptitle(f'Test Set Error Statistics  (N={N_TEST})', color='#e6edf3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_error_distribution.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("Saved fig4_error_distribution.png")


In [ ]:
# Save model and download figures
model.save('magcnn_model.keras')
print("Model saved: magcnn_model.keras")

# Download all figures to your local machine
from google.colab import files
for fname in ['fig1_training_curves.png', 'fig2_reconstruction_examples.png',
              'fig3_domain_reconstruction.png', 'fig4_error_distribution.png',
              'magcnn_model.keras']:
    try:
        files.download(fname)
        print(f"  Downloaded: {fname}")
    except:
        print(f"  {fname} ready in file browser (left panel)")
